# DoE analysis to identify the contribution of each feature

The expected format of our input CSVs is a set of columns with 1s and -1s and a set of columns with numerical values between 0 and 1
(they represent the corresponding Accuracy, Precision and Recall for every run).
The repetitions parameter is requested as a variable in the functions and represents how many times each combination of our DoE
was run.
In our input csv every repetition comes after another, so if we had two combinations and 5 repetitions each, we would have ten rows in our csv, the first 5 for the 5 runs of the first combination and the next 5 for the 5 runs of the second combination.

### CSVs

In [1]:
#This first csv contains the DoE
csv_name = '1 and -1 Sorted.csv'
#This csv will contain the result of the analysis of the influence of each feature
name_return_csv = 'test2'

### Imports

In [2]:
#Here are the necessary imports
import pandas as pd
import numpy as np
import itertools
import csv

### Run the code

In [3]:
#Run the following if your input csv has more than 1 evaluation metric
# return_csv(csv_name,name_return_csv, repetitions)

#Run the following if your input csv has exactly 1 evaluation metric
# return_csv_2(csv_name,name_return_csv, repetitions)



# Code 

### Read and parse CSV

In [4]:
# Function to read a CSV file and convert it to a pandas DataFrame
def read_csv_to_df(csv_file):
        df = pd.read_csv(csv_file) # Read the CSV file and store it in a DataFrame
        return df    # Return the DataFrame

# Function to read and parse a CSV file
def read_and_parse_csv(csv_name, repetitions):
    csv_file = csv_name # Name of the CSV file
    df = read_csv_to_df(csv_file) # Read the CSV file and get the DataFrame

    # Extract only numerical columns with all values strictly under 2
    numerical_df = df.select_dtypes(include=[np.number]) # Select only numerical columns
    filtered_numerical_df = numerical_df.loc[:, (numerical_df < 2).all()] # Keep columns where all values are < 2

    # Group by every 5 rows and calculate the mean
    avg_df = filtered_numerical_df.groupby(df.index // repetitions).mean().reset_index(drop=True)

    # Separate columns into those that have -1 as a value and those that don't 
    # to get the columns with the binary options in the DoE and the columns with the values for the metrics
    columns_with_minus_one = avg_df.loc[:, (filtered_numerical_df == -1).any()] # Columns with at least one -1
    columns_without_minus_one = avg_df.loc[:, ~(filtered_numerical_df == -1).any()] # Columns without -1
    
    # Return the resulting DataFrames
    return avg_df, columns_with_minus_one, columns_without_minus_one

### Compute the 2^k factorial analysis with singletons and pairs of features

In [5]:
def create_factorial_design(csv_name, repetitions):
    # Read and parse the CSV file to get the average DataFrame and the specific columns
    avg_df, columns, specific_columns = read_and_parse_csv(csv_name, repetitions) 
    # avg_df = Dataframe containing the average for each set of n repetitions
    # columns = Dataframe containing the binary features for each prompt combination
    # specific_columns = Dataframe containing the metrics for each prompt combination

    # Multiply each column containing the binary features with each column containing the metrics
    # To obtain the singletons for the features in the 2^k factorial analysis
    for col in columns:
        for spec_col in specific_columns:
            new_col_name = f"{col}_evaluating_{spec_col}" # Indicate the name of which feature is being multiplied to which metric
            avg_df[new_col_name] = avg_df[col] * avg_df[spec_col]

    # Loop through combinations of exactly two features and calculate the product with each column containing the result for each metric
    # To obtain the pairs for the features in the 2^k factorial analysis
    for combination in itertools.combinations(columns, 2):
        col_name = '_AND_'.join(combination) # Indicate the name of which pair of features is being created
        avg_df[col_name] = avg_df[list(combination)].prod(axis=1)
        # Multiply with each of the columns containg the values for the metrics 
        for spec_col in specific_columns:
            new_col_name = f"{col_name}_evaluating_{spec_col}" # Indicate which pair of features is multiplied to which metric
            avg_df[new_col_name] = avg_df[col_name] * avg_df[spec_col]
            
    return avg_df, columns, specific_columns

### Compute the SST and the sums for each column of the 2^k factorial analysis

In [6]:
# Function to process the sums and SST for each metric
def calculate_sums_and_sst(df, metric, num_of_experiments):
    columns = df.filter(like=metric).columns
    metric_df = df[columns]
    # Compute the sum of each feature combination column
    column_sums = [(col, metric_df[col].sum() / (2**num_of_experiments)) for col in metric_df.columns] 
    # Application of the formula to compute the SST
    sst = sum([col_sum[1]**2 for col_sum in column_sums[1:]]) * (2**num_of_experiments)
    
    return column_sums, sst

# Function to add the SST for each metric to the list containing the results of the sums for each combination of metrics considered 
def calculate_sst(csv_name, repetitions):
    # Obtention of the necessary elements with previous fonctions
    list_of_name_and_sst=[]
    avg_df, columns, specific_columns = create_factorial_design(csv_name, repetitions)
    num_of_experiments = columns.shape[1] 
    
    # Adding SST for each metric to the list
    for column in specific_columns:
        name = 'column_sums_'+column
        sst = 'sst_'+column
        name, sst = calculate_sums_and_sst(avg_df, column, num_of_experiments) # Computation
        list_of_name_and_sst.append((name,sst)) # Append to the list
        
    # Return the list containing the sums of each column and the SST for each metric
    return list_of_name_and_sst, num_of_experiments

### Compute the contribution of each combination of features considered to each metric

In [7]:
def compute_results(csv_name, repetitions):
    # Obtention of previous results
    list_of_name_and_sst, num_of_experiments = calculate_sst(csv_name, repetitions)
    
    # Create a list to store the contribution of each combination of features considered to every evaluated metric
    list_with_sst=[]
    for j in range(len(list_of_name_and_sst)):
        list_with_sst_2=[]
        for i in range(1,len(list_of_name_and_sst[j][0])):
            # Application of the formula to compute the % of contribution 
            list_with_sst_2.append((list_of_name_and_sst[j][0][i][0],(list_of_name_and_sst[j][0][i][1]**2)*(2**num_of_experiments)*100/list_of_name_and_sst[j][1]))
        list_with_sst.append(list_with_sst_2)
        
    return list_with_sst

### Generation of the CSV with the contribution of each combination of features considered to every evaluated metric

In [8]:
# Optional function that removes the word 'evaluating' from the names of columns, to improve the readability if needed
def remove_evaluating_suffix(input_string):
    # Find the position of "_evaluating"
    position = input_string.find("_evaluating")
    # If "_evaluating" is found, return the substring before it
    if position != -1:
        return input_string[:position]
    # If "_evaluating" is not found, return the original string
    return input_string

# Generation of the CSV with 2 columns, one indicating the analysed combinations, the other containing the contribution 
# In the case we have more than 1 evaluated metric
def return_csv(csv_name, name_return_csv, repetitions):
    list_with_sst = compute_results(csv_name, repetitions)
        # File name
    filename = name_return_csv+'.csv'

    # Writing to the CSV file
    with open(filename, 'w', newline='') as csvfile:
        # Create a csv writer object
        csvwriter = csv.writer(csvfile)
        list_of_titles=['Feature']
        for i in range(len(list_with_sst)):
            parts = list_with_sst[i][0][0].split('_')
            title= parts[-1]
            list_of_titles.append(title)
        csvwriter.writerow(list_of_titles)

        # Write the list to the csv file
        rows=[]
        for k in range(len(list_with_sst[1])):
            list_for_row = []
            output_string = remove_evaluating_suffix(list_with_sst[1][k][0])
            list_for_row.append(output_string)
            for j in range(len(list_with_sst)):
                list_for_row.append(list_with_sst[j][k][1])
            rows.append(list_for_row)

        for row in rows:
            csvwriter.writerow(row)

    print(f"List written to {filename} successfully.")
    
# Generation of the CSV with 2 columns, one indicating the analysed combinations, the other containing the contribution 
# In the case we have only 1 evaluated metric
def return_csv_2(csv_file,name_return_csv, repetitions):
    data = compute_results(csv_file, repetitions)[0]
    filtered_data = data

    # Remove '_evaluating' from each feature name
    processed_data = [(feature.replace('_evaluating', ''), result) for feature, result in filtered_data]

    # Specify the filename
    filename = name_return_csv+'.csv'

    # Write the processed data into a CSV file with columns 'feature' and 'result'
    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Feature', 'Result'])
        writer.writerows(processed_data)

    print(f"List written to {filename} successfully.")    

In [9]:
return_csv(csv_name, name_return_csv, 3)

List written to test2.csv successfully.
